In [9]:
import pandas as pd
import io

In [10]:
data_referencia = '28-02-2026'

In [11]:
df_baixas = pd.read_excel('../data/raw/finr190.xlsx', sheet_name='Baixas', skiprows=1)

In [12]:
df_titulos = pd.read_excel('../data/raw/finr130.xlsx', sheet_name='Titulos a receber')

In [13]:
df_titulos.columns = [col.replace('_x000D_\n', ' ') for col in df_titulos.columns]

In [14]:
df_totais = pd.read_excel('../data/raw/finr130.xlsx', sheet_name='Totais', skiprows=1)

In [15]:
df_totais.columns = [col.replace('_x000D_\n', ' ') for col in df_totais.columns]

In [16]:
df_ra = pd.read_excel('../data/raw/fina740.xlsx', skiprows=1)

In [17]:
df_resumo = pd.DataFrame(columns= ['PRT','MOT','Valor Original','Jur/Multa','Correcao','Descontos','Abatimentos','Impostos','Valor Acessorio','Total Baixado'])

In [18]:
resumo_rows = []

for (prt, mot), grupo in df_baixas.groupby(["Prf", "Mot"]):
    resumo_rows.append({
        "PRT": prt,
        "MOT": mot,
        "Valor Original": grupo["Valor Original"].sum(),
        "Jur/Multa": grupo["Jur/Multa"].sum(),
        "Correcao": grupo["Correcao"].sum(),
        "Descontos": grupo["Descontos"].sum(),
        "Abatimentos": grupo["Abatim."].sum(),
        "Impostos": grupo["Impostos"].sum(),
        "Valor Acessorio": grupo["Valor Acessorio"].sum(),
        "Total Baixado": grupo["Total Baixado"].sum(),
    })

df_resumo = pd.DataFrame(resumo_rows)

In [19]:
d = {
    'Recebimento de Títulos':float(df_resumo[(df_resumo.PRT == '4') & (df_resumo.MOT == 'CMP')]['Total Baixado'].iloc[0].round(2)),
    'Retenção de Tributos':float(df_resumo[(df_resumo.PRT == '4') & (df_resumo.MOT == 'CMP')]['Impostos'].iloc[0].round(2)),
    'Compensações de Títulos':float(df_resumo[(df_resumo.PRT == '4') & (df_resumo.MOT == 'NOR')]['Total Baixado'].iloc[0].round(2)),
    f"Saldo a receber em {data_referencia}":float(df_totais.drop(df_totais.index[-1])['(Vencidos+Vencer)'].sum().round(2)),
    f"Saldo RA em {data_referencia}":float(df_ra.Saldo.sum().round(2))
}

In [20]:
df_relatorio = pd.DataFrame([d])

In [21]:
with pd.ExcelWriter(f"relatorio-clubedamedalha-{data_referencia}.xlsx", engine='openpyxl') as writer:
    df_baixas.to_excel(writer, sheet_name='Baixas', index=False)
    for x in df_baixas.Prf.unique():
        for y  in df_baixas.Mot.unique():
            df_baixas[(df_baixas.Prf == x) & (df_baixas.Mot == y)].to_excel(writer, sheet_name=f"{x} {y}", index=False)
    df_totais.to_excel(writer, sheet_name='Saldo a receber', index=False)
    df_ra.to_excel(writer, sheet_name='Saldo RA', index=False)
    df_resumo.to_excel(writer, sheet_name='Resumo', index=False)
    df_relatorio.to_excel(writer, sheet_name='Relatorio', index=False)